# feature 정보 활용 제주 버스 도착 시간 예측

제주 버스 도착 시간 예측 프로젝트를 통해 **기준 모델 설정 → 오차 분석 → feature 실험 → 결과 비교**까지,
머신러닝 실험의 전체 흐름을 단계별로 학습하는 노트북입니다.

## 이번 노트북의 전체 실험 흐름
1. 데이터 불러오기
2. 날짜·시간 feature 생성
3. 정류장 구간 feature 생성 (`station_segment`)
4. 위치 권역 feature 생성 (`dist_segment_name`)
5. 기준 모델 `v2_original` 학습
6. 기준 모델 오차 분석
7. target 상위 1% 제외 실험
8. `station_segment` 추가 실험
9. `dist_segment_name` 추가 실험
10. 결론 정리

> 이 노트북에서는 결측치 처리, 복잡한 EDA, 하이퍼파라미터 튜닝, XGBoost 이론 심화는 다루지 않습니다.
> feature 실험 설계 → 오차 분석 → 실험 결과 비교 및 해석에 집중합니다.


## 1. 라이브러리 불러오기

In [ ]:
# 이 노트북을 실행하려면 xgboost 패키지가 필요합니다.
# 설치가 안 되어 있으면 아래 명령을 노트북 셀에서 한 번 실행하세요.
# !pip install xgboost
# uv 환경에서는 터미널에서: uv add xgboost

import pandas as pd                # CSV 파일을 읽고 표(DataFrame) 형태 데이터를 다루는 라이브러리
import numpy as np                 # 수치 계산용 라이브러리 (거리 계산, RMSE 계산 등에 사용)
import matplotlib.pyplot as plt    # 오차 분포, 성능 비교 등을 그래프로 그리는 라이브러리
import warnings                    # 실행 중 나오는 경고 메시지를 정리하기 위한 표준 라이브러리

# scikit-learn: 데이터 분리·전처리·평가 지표를 제공하는 머신러닝 도구 모음
from sklearn.model_selection import train_test_split      # 데이터를 학습용/검증용으로 나누는 함수
from sklearn.compose import ColumnTransformer              # 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.pipeline import Pipeline                       # 전처리와 모델 학습을 하나로 묶는 도구
from sklearn.preprocessing import OneHotEncoder             # 문자(범주형) 값을 0/1 숫자 컬럼으로 바꾸는 도구
from sklearn.metrics import mean_absolute_error, mean_squared_error  # 회귀 평가 지표
from xgboost import XGBRegressor                            # 이번에 사용할 예측 모델 (회귀용 XGBoost)

# 안내성 경고 메시지를 숨겨 수업 흐름의 가독성을 높입니다. (오류가 아니라 참고용 경고입니다.)
warnings.filterwarnings("ignore")
plt.rcParams["axes.unicode_minus"] = False  # 그래프에서 음수 부호(-)가 깨지지 않도록 하는 설정

plt.style.use("ggplot")


## 2. 데이터 불러오기와 기본 확인

데이터 분석의 첫 단계는 데이터를 불러오고 구조를 확인하는 것입니다. 이 과정은 단순해 보이지만, 이후 모든 분석의 기반이 됩니다.

이번 데이터에는 결측치가 없으므로 별도의 결측치 대체 과정은 생략합니다. 이 노트북의 목적은 결측치 처리가 아니라 feature 실험과 오차 분석입니다.


In [ ]:
# pd.read_csv() 는 CSV 파일을 읽어 표(DataFrame) 형태로 만들어 주는 함수입니다.
# 노트북과 같은 폴더에 jeju_bus.csv 가 있다고 가정합니다.
df = pd.read_csv("jeju_bus.csv")

# df.head(): 데이터의 맨 위 5개 행을 보여 줍니다. 값이 어떤 형태인지 눈으로 확인합니다.
display(df.head())

# df.shape: (행 개수, 열 개수)를 알려 줍니다. 데이터가 몇 건이고 컬럼이 몇 개인지 파악합니다.
print("shape:", df.shape)

# 결측치(비어 있는 값)가 있으면 전처리가 필요하지만, 이 데이터에는 결측치가 없습니다.
# 전체 결측치 개수가 0 이면 별도의 결측치 처리 없이 바로 feature 실험에 집중할 수 있습니다.
print("전체 결측치 개수:", df.isnull().sum().sum())


## 3. 원본 보존과 모델링용 데이터

노트북에서는 원본 데이터 `df`를 직접 수정하지 않고, 복사본 `df_model`을 만들어 사용합니다. 이 습관은 실무에서 매우 중요합니다.

- **df (원본)**: 절대 수정하지 않음. 언제든 되돌릴 수 있는 안전망.
- **df_model (복사본)**: feature 추가, 이상치 제거 등 모든 실험에 사용.
- **original_index (연결고리)**: 예측 결과와 원본 행을 연결하기 위한 인덱스 보관.


In [ ]:
# 원본 df 는 그대로 보존하고, 분석·모델링은 복사본 df_model 에서만 진행합니다.
# (원본을 직접 고치면 중간에 문제가 생겼을 때 처음 상태로 되돌리기 어렵기 때문입니다.)
df_model = df.copy()

# original_index: 각 행의 원래 번호를 따로 저장해 둡니다.
# 학습/검증으로 데이터를 나눈 뒤에도, 예측 결과를 원본 행과 다시 연결해 오차를 분석하기 위해서입니다.
df_model["original_index"] = df_model.index

# target_col: 우리가 맞히려는 정답 컬럼입니다. (다음 정류장까지 걸린 시간)
target_col = "next_arrive_time"  # 예측 대상(정답)


## 4. 날짜·시간 Feature 생성

버스 도착 시간은 시간대에 따라 달라질 수 있습니다. 출근 시간, 퇴근 시간, 낮 시간, 밤 시간의 교통 상황은 서로 다를 수 있기 때문입니다. 이 가정을 모델에 반영하기 위해 날짜·시간 feature를 만듭니다.

- **day**: 월 중 며칠인지 (1~31)
- **dayofweek**: 요일 (월=0 ... 일=6)
- **now_hour**: 현재 정류장 도착 시간의 시(hour) (0~23)


In [ ]:
# 날짜가 "2019-10-15" 같은 문자열이면 모델이 이해하기 어렵습니다.
# pd.to_datetime() 으로 날짜형(datetime)으로 바꾸면 연/월/일/요일을 바로 뽑아낼 수 있습니다.
df_model["date"] = pd.to_datetime(df_model["date"])

# 날짜에서 모델이 활용할 숫자 feature 를 만듭니다.
df_model["day"] = df_model["date"].dt.day              # 며칠인지 (1~31)
df_model["dayofweek"] = df_model["date"].dt.dayofweek  # 요일 (월=0 ... 일=6). 평일/주말 패턴을 반영할 수 있습니다.

# now_arrive_time 은 "06시"처럼 시(hour)만 담긴 문자열입니다. 여기서 숫자 시간대(now_hour)를 뽑습니다.
# 출근·퇴근 시간대에는 교통 상황이 달라질 수 있으므로, 시간대는 도착 시간 예측에 중요한 feature 가 될 수 있습니다.
df_model["now_hour"] = df_model["now_arrive_time"].astype(str).str.extract(r"(\d+)").astype(float)

# 만들어진 시간 feature 를 눈으로 확인합니다.
df_model[["date", "day", "dayofweek", "now_arrive_time", "now_hour"]].head()


## 5. 정류장 구간 Feature: station_segment

기존 데이터에는 현재 정류장 `now_station`과 다음 정류장 `next_station`이 따로 있습니다. 하지만 버스 도착 시간은 현재 정류장만으로 결정되지 않습니다. 현재 정류장에서 다음 정류장까지의 구간 특성이 중요할 수 있습니다.

> ⚠️ 정류장 구간은 종류가 매우 많을 수 있습니다. 범주형 feature는 One-Hot Encoding을 거치면 여러 개의 숫자 컬럼으로 바뀌는데, 고유값이 많을수록 생성되는 컬럼 수도 많아집니다.


In [ ]:
# 현재 정류장 이름과 다음 정류장 이름을 " → " 로 이어 붙여 "구간" feature 를 만듭니다.
# 도착 시간은 현재 정류장만이 아니라, 현재→다음 정류장 구간의 특성에 영향을 받을 수 있기 때문입니다.
df_model["station_segment"] = (
    df_model["now_station"].astype(str) + " → " + df_model["next_station"].astype(str)
)

df_model.head()


In [ ]:
# nunique(): 서로 다른 값이 몇 종류인지 셉니다. 값이 많을수록 이후 One-Hot 컬럼 수도 많아집니다.
print("station_segment 고유값 개수:", df_model["station_segment"].nunique())


## 6. Haversine 거리 계산: 왜 특별한 공식이 필요한가?

위도와 경도는 평면 좌표가 아니라 지구 표면 위의 좌표입니다. 따라서 두 좌표 사이의 거리를 계산할 때는 일반적인 직선 거리 계산과 다른 방식이 필요합니다.

이 노트북에서는 `numpy` 기반 vectorized 계산을 사용합니다. 한 행씩 반복해서 계산하는 것이 아니라, 컬럼 전체를 한 번에 계산하는 방식이라 데이터가 많을 때 훨씬 효율적입니다.

> 핵심은 공식을 외우는 것이 아니라, "위도/경도 좌표를 이용해 거리 기반 feature를 빠르게 만든다"는 것입니다.


In [ ]:
def calculate_distance_km(lat1, lon1, lat2, lon2):
    """
    위도/경도 사이의 거리를 km 단위로 계산합니다.

    geopy.distance.distance()는 사용하기 쉽지만, 데이터가 많을 때 행마다 반복 호출하면
    속도가 느릴 수 있습니다. 그래서 여기서는 numpy 기반 Haversine 계산을 사용합니다.

    이 함수는 단일 값뿐 아니라 pandas Series 또는 numpy array도 받을 수 있습니다.
    따라서 apply(axis=1) 없이 컬럼 전체를 한 번에 계산할 수 있습니다.
    수식을 외우는 것이 목적은 아니며, 좌표로 거리 feature를 빠르게 만든다는 점이 중요합니다.
    """
    earth_radius_km = 6371

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c


## 7. 위치 권역 Feature: dist_segment_name

정류장 이름은 매우 구체적인 정보입니다. 반면 위치 권역은 정류장을 조금 더 넓은 지역 단위로 묶은 정보입니다. 이 feature는 지리적 흐름을 모델에 알려주기 위한 것입니다.

- **now_dist_name**: 현재 정류장이 가까운 권역 이름
- **next_dist_name**: 다음 정류장이 가까운 권역 이름
- **dist_segment_name**: 현재 권역에서 다음 권역으로 이동하는 구간 이름 (예: 제주시 → 서귀포시)

> ⚠️ 이미 now_station, next_station, route_nm에 위치 정보가 어느 정도 포함되어 있을 수 있습니다. 이 경우 dist_segment_name은 기존 feature와 정보가 중복될 수 있습니다. 따라서 결과는 반드시 비교표를 통해 확인해야 합니다.


In [ ]:
# 제주도 내 주요 기준 지점 4개 (위도, 경도)
reference_points = {
    "up": (33.506286, 126.490312),      # 제주공항
    "down": (33.246742, 126.562387),    # 서귀포시 근처
    "right": (33.493521, 126.895326),   # 성산일출봉 방면
    "center": (33.379724, 126.545315),  # 한라산 / 중산간
}


In [ ]:
def assign_region_info(data, lat_col, lon_col):
    """
    각 정류장의 좌표를 기준으로 4개 기준점까지의 거리를 계산하고,
    가장 가까운 권역 이름과 기준점별 거리 컬럼을 함께 반환합니다.

    반환되는 컬럼:
    - dist_name: 가장 가까운 권역 이름
    - dist_to_up: up 기준점까지의 거리
    - dist_to_down: down 기준점까지의 거리
    - dist_to_right: right 기준점까지의 거리
    - dist_to_center: center 기준점까지의 거리
    """
    # 1. 각 기준점의 위도/경도 값을 꺼냅니다.
    up_lat, up_lon = reference_points["up"]
    down_lat, down_lon = reference_points["down"]
    right_lat, right_lon = reference_points["right"]
    center_lat, center_lon = reference_points["center"]

    # 2. 각 정류장에서 4개 기준점까지의 거리를 계산합니다.
    distance_to_up = calculate_distance_km(data[lat_col], data[lon_col], up_lat, up_lon)
    distance_to_down = calculate_distance_km(data[lat_col], data[lon_col], down_lat, down_lon)
    distance_to_right = calculate_distance_km(data[lat_col], data[lon_col], right_lat, right_lon)
    distance_to_center = calculate_distance_km(data[lat_col], data[lon_col], center_lat, center_lon)

    # 3. 가장 가까운 권역 이름을 찾기 위한 거리표를 만듭니다.
    # 여기서는 컬럼명을 up, down, right, center로 묶어
    # idxmin(axis=1)의 결과가 바로 권역 이름이 됩니다.
    distance_table = pd.DataFrame({
        "up": distance_to_up,
        "down": distance_to_down,
        "right": distance_to_right,
        "center": distance_to_center,
    }, index=data.index)

    # 4. 각 행에서 가장 가까운 기준점 이름을 찾습니다.
    nearest_region = distance_table.idxmin(axis=1)

    # 5. 최종 반환용 DataFrame을 만듭니다.
    # 거리 컬럼명은 모델 feature로 쓰기 좋게 명확하게 작성합니다.
    result = pd.DataFrame({
        "dist_name": nearest_region,
        "dist_to_up": distance_to_up,
        "dist_to_down": distance_to_down,
        "dist_to_right": distance_to_right,
        "dist_to_center": distance_to_center,
    }, index=data.index)

    return result


In [ ]:
# 현재 정류장 기준 권역 정보 계산
now_region_info = assign_region_info(
    df_model,
    "now_latitude",
    "now_longitude"
)

now_region_info.head()


In [ ]:
# 다음 정류장 기준 권역 정보 계산
next_region_info = assign_region_info(
    df_model,
    "next_latitude",
    "next_longitude"
)

next_region_info.head()


In [ ]:
# 현재 정류장 권역 이름과 거리 저장
df_model["now_dist_name"] = now_region_info["dist_name"]
df_model["now_dist_to_up"] = now_region_info["dist_to_up"]
df_model["now_dist_to_down"] = now_region_info["dist_to_down"]
df_model["now_dist_to_right"] = now_region_info["dist_to_right"]
df_model["now_dist_to_center"] = now_region_info["dist_to_center"]

df_model.head()


In [ ]:
# 다음 정류장 권역 이름과 거리 저장
df_model["next_dist_name"] = next_region_info["dist_name"]
df_model["next_dist_to_up"] = next_region_info["dist_to_up"]
df_model["next_dist_to_down"] = next_region_info["dist_to_down"]
df_model["next_dist_to_right"] = next_region_info["dist_to_right"]
df_model["next_dist_to_center"] = next_region_info["dist_to_center"]

df_model.head()


In [ ]:
# 현재 정류장 권역과 다음 정류장 권역을 조합한 feature 생성
df_model["dist_segment_name"] = (
    df_model["now_dist_name"].astype(str)
    + " → "
    + df_model["next_dist_name"].astype(str)
)

# 조합 권역이 몇 종류인지, 어떤 조합이 많은지 확인
print("dist_segment_name 고유값 개수:", df_model["dist_segment_name"].nunique())
df_model["dist_segment_name"].value_counts()


## 8. train/test split의 의미

머신러닝에서는 데이터를 보통 학습용 데이터와 테스트용 데이터로 나눕니다.

> 이번 노트북에서는 여러 실험을 공정하게 비교하기 위해 같은 train/test split을 사용합니다. feature만 바꾸고 데이터 분할 조건은 유지합니다. 그래야 성능 차이가 데이터 분할 때문인지 feature 때문인지 혼동을 줄일 수 있습니다.


In [ ]:
# df_model.index 에서 몇번째 행을 학습데이터로 테스트 데이터로 할지 정함
train_idx, test_idx = train_test_split(df_model.index, test_size=0.2, random_state=42)


In [ ]:
# 학습 데이터 줄 번호
train_idx


In [ ]:
# 테스트 데이터 줄 번호
test_idx


In [ ]:
print("train 행 수:", len(train_idx), "/ test 행 수:", len(test_idx))


## 9. 평가 지표: MAE, RMSE

- **MAE (Mean Absolute Error)**: 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균을 낸 값입니다. "평균적으로 몇 초 정도 틀렸는가"를 보는 지표입니다. 작을수록 좋습니다.
- **RMSE (Root Mean Squared Error)**: 큰 오차에 더 민감하게 반응하는 지표입니다. 모델이 가끔 크게 틀리는 경우 RMSE가 커질 수 있습니다. 작을수록 좋습니다.

> 세 지표를 함께 보는 이유: MAE는 평균적인 오차를, RMSE는 큰 오차의 영향을 나타냅니다. 하나의 지표만으로는 모델의 전체 상태를 파악하기 어렵습니다.


In [ ]:
def evaluate_regression_model(model_name, experiment, y_true, y_pred,
                                directly_comparable=True, note=""):
    """
    회귀 모델의 예측 결과를 평가해 딕셔너리로 돌려주는 함수입니다.
    여러 실험을 같은 방식으로 평가하려고 함수로 묶었습니다.

    - MAE : 평균적으로 몇 초 정도 틀렸는지를 보는 지표 (작을수록 좋음)
    - RMSE: 큰 오차에 더 민감한 지표. 크게 틀린 예측이 많으면 값이 커집니다 (작을수록 좋음)
    여러 지표를 함께 보는 이유는, 한 지표만으로는 모델의 약점을 놓칠 수 있기 때문입니다.

    directly_comparable: 이 모델을 기준 모델과 '직접 점수 비교'해도 되는지 표시합니다.
      평가에 사용한 데이터(test set)가 다르면 단순 비교가 공정하지 않으므로 False 로 둡니다.
    """
    mae = mean_absolute_error(y_true, y_pred)
    # RMSE 는 일부 버전에서 squared=False 옵션이 없으므로, 호환성을 위해 np.sqrt 로 직접 계산합니다.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"[{model_name}] MAE: {mae:.4f} | RMSE: {rmse:.4f} ")
    # 나중에 여러 모델 결과를 한 표로 모으기 좋도록 딕셔너리로 반환합니다.
    return {
        "model_name": model_name, "experiment": experiment,
        "directly_comparable": directly_comparable,
        "MAE": mae, "RMSE": rmse, "note": note,
    }


## 10. 숫자형 Feature와 범주형 Feature, XGBoost 학습 함수

머신러닝 모델에 입력되는 데이터는 대부분 숫자 형태여야 합니다.

- **숫자형 feature** (거리, 날짜의 일, 요일, 시간대 등): 이미 숫자로 표현됨 → 그대로 사용 가능
- **범주형 feature** (노선명, 현재 정류장명, 다음 정류장명 등): 종류나 이름을 나타내는 문자 → One-Hot Encoding으로 숫자 컬럼으로 변환 필요

`handle_unknown="ignore"` 옵션은 학습 데이터에는 없던 정류장명이나 노선명이 테스트 데이터에 등장해도 오류를 내지 않고 처리하도록 하는 설정입니다. 결측치 처리가 아니라, 학습/테스트의 범주 종류가 다를 때 발생할 수 있는 문제를 막기 위한 설정입니다.

아래 `train_xgb` 함수는 주어진 feature 목록으로 XGBoost 회귀 모델을 학습하고 평가하는 함수입니다. feature 목록만 바꿔 여러 번 호출하면, 그 차이에 의한 성능 변화를 비교할 수 있습니다.

XGBoost는 여러 개의 작은 결정트리를 차례로 만들며 앞 트리의 오차를 보정해 나가는 회귀 모델입니다. 이번 노트북은 feature 효과 비교가 목적이므로 하이퍼파라미터는 고정합니다. (하이퍼파라미터 튜닝은 다음 노트북에서 다룹니다.)


In [ ]:
#train_xgb(model_name, experiment, numeric_features, categorical_features,
#           train_idx, test_idx, data_df, directly_comparable=True, note=""):
#
# 매개변수 의미
# - model_name        : 결과표에 표시할 모델 이름 (예: "v2_original")
# - experiment        : 어떤 feature 구성을 썼는지에 대한 짧은 설명
# - numeric_features  : 숫자형 feature 목록 (그대로 모델에 전달)
# - categorical_features: 범주형(문자) feature 목록 (One-Hot Encoding 후 전달)
# - train_idx / test_idx: 학습용/검증용으로 쓸 행 번호. 모든 모델이 같은 split 을 써야 공정한 비교가 됩니다.
# - data_df           : 사용할 데이터프레임 (이상치 제외 실험에서는 다른 데이터를 넘깁니다.)

def train_xgb(model_name, experiment, numeric_features, categorical_features,
              train_idx, test_idx, data_df, directly_comparable=True, note=""):

    # 사용할 feature 만 모아 입력 X 와 정답 y 를 만듭니다.
    feats = numeric_features + categorical_features
    X, y = data_df[feats], data_df[target_col]
    # 같은 행 번호(train_idx/test_idx)로 학습용/검증용을 나눕니다.
    # X_train/y_train 으로 학습하고, 모델이 보지 못한 X_test/y_test 로 성능을 확인합니다.
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    # ColumnTransformer: 컬럼 종류별로 다른 전처리를 한 번에 적용하는 도구입니다.
    # - 범주형 feature 는 OneHotEncoder 로 0/1 숫자 컬럼으로 변환합니다.
    #   handle_unknown="ignore": 검증/실제 데이터에 학습 때 못 본 새 범주가 와도 오류 없이 처리합니다.
    # - remainder="passthrough": 나머지 숫자형 feature 는 변환 없이 그대로 모델에 전달합니다.
    #   (XGBoost 같은 트리 모델은 숫자 크기 스케일을 맞출 필요가 없기 때문입니다.)
    preprocessor = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="passthrough",
    )

    # 모든 모델이 똑같이 사용하는 XGBoost 설정입니다. (feature 효과만 비교하려고 고정)
    # - n_estimators : 만들 트리 개수
    # - learning_rate: 한 번에 얼마나 조금씩 보정할지 (작을수록 신중하게 학습)
    # - max_depth    : 각 트리의 최대 깊이 (클수록 복잡한 패턴을 학습)
    # - random_state : 결과 재현을 위한 난수 고정값
    # - objective    : 회귀(제곱 오차를 줄이는 방향)로 학습한다는 설정
    model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5,
                          random_state=42, objective="reg:squarederror")

    # 전처리 -> 모델 학습을 하나의 흐름(Pipeline)으로 묶습니다. (예측 때도 같은 전처리가 자동 적용)
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)          # fit(): 입력(X_train)과 정답(y_train)의 관계를 학습
    y_pred = pipe.predict(X_test)       # predict(): 학습한 모델로 검증 데이터의 도착 시간을 예측

    # 예측 결과를 평가해 딕셔너리로 정리합니다.
    evaluation = evaluate_regression_model(model_name, experiment, y_test, y_pred,
                                            directly_comparable=directly_comparable, note=note)
    # 이후 오차 분석에서 다시 쓰도록 파이프라인·평가결과·예측값을 함께 돌려줍니다.
    return {"pipeline": pipe, "evaluation": evaluation, "y_test": y_test, "y_pred": y_pred}


## 11. 기준 모델 v2_original: 왜 기준 모델이 필요한가?

기준 모델은 이후 실험을 비교하기 위한 출발점입니다. feature를 새로 추가했을 때 성능이 좋아졌는지 나빠졌는지 판단하려면 먼저 기준이 있어야 합니다.

- **숫자형 feature (그대로 사용 가능)**: `distance`, `day`, `dayofweek`, `now_hour`
- **범주형 feature (인코딩 필요)**: `route_nm`, `now_station`, `next_station`


In [ ]:
# 기준 모델 v2_original 의 feature 목록입니다.
# 숫자형(거리·시간)과 범주형(노선·정류장)을 나누는 이유는, 범주형만 One-Hot Encoding 으로
# 숫자 컬럼으로 바꿔야 하기 때문입니다. (train_xgb 함수가 두 목록을 다르게 처리합니다.)
numeric_features_v2 = ["distance", "day", "dayofweek", "now_hour"]
categorical_features_v2 = ["route_nm", "now_station", "next_station"]

# 기준 모델을 학습합니다. 이후 실험들은 이 모델과 비교할 출발점이 됩니다.
v2_result = train_xgb(
    model_name="v2_original", experiment="EDA 기반 v2 feature",
    numeric_features=numeric_features_v2, categorical_features=categorical_features_v2,
    train_idx=train_idx, test_idx=test_idx, data_df=df_model, note="기준 모델",
)


## 12. 오차 분석이 필요한 이유

모델 평가에서 평균 성능 지표는 중요합니다. 하지만 평균만 보면 모델이 어떤 상황에서 약한지 알기 어렵습니다. 전체 평균 오차는 작아도, 특정 시간대나 특정 구간에서는 오차가 매우 클 수 있습니다.

- **actual**: 실제 도착 시간 (정답)
- **pred**: 모델이 예측한 도착 시간
- **error**: 예측값 - 실제값 (부호 있음)
- **abs_error**: 오차의 절댓값 (크기만 봄)


In [ ]:
# 평균 지표만으로는 모델이 "어디서" 틀리는지 알 수 없습니다.
# 그래서 검증 데이터의 실제값/예측값/오차를 행 단위로 모아 봅니다.
# original_index 를 만들어 두었기 때문에, 검증 행을 원본 df_model 에서 그대로 가져올 수 있습니다.
err = df_model.loc[test_idx].copy()
err["actual"] = v2_result["y_test"].values             # actual: 실제 도착 시간(정답)
err["pred"] = v2_result["y_pred"]                       # pred: 모델이 예측한 도착 시간
err["abs_error"] = np.abs(err["actual"] - err["pred"])  # abs_error: 오차의 절댓값(얼마나 벗어났는지)

# 절대 오차가 큰 상위 20건 - 모델이 가장 크게 틀린 사례입니다.
# 어떤 노선·정류장·시간대·거리에서 어려워하는지 힌트를 얻을 수 있습니다. (단정이 아니라 힌트입니다.)
err.sort_values("abs_error", ascending=False)[
    ["actual", "pred", "abs_error", "route_nm", "now_station", "next_station", "now_hour", "distance"]
].head(20)


### 오차가 큰 데이터 확인: 모델의 약점 찾기

오차가 큰 데이터 상위 행을 확인하는 이유는 모델의 약점을 찾기 위해서입니다. 모델이 크게 틀린 행을 보면 다음과 같은 질문을 던질 수 있습니다.

- 특정 노선에서 오차가 큰가?
- 특정 정류장 구간에서 오차가 큰가? (`station_segment` 같은 구간 feature를 추가해 볼 수 있습니다)
- 특정 시간대에서 오차가 큰가? (출퇴근 시간대에 교통 상황이 복잡해 예측이 어려울 수 있습니다)
- 실제 도착 시간이 매우 긴 이상치인가? (모델이 학습하기 어려운 패턴일 수 있습니다)


## 13. 오차 분포 확인

오차 분석에서는 오차의 분포도 확인합니다. 단순히 평균 숫자를 보는 것보다 모델의 상태를 더 잘 이해하는 데 도움이 됩니다.


In [ ]:
# abs_error 의 분포를 봅니다. 대부분의 오차가 작은 구간에 몰려 있는지,
# 일부 아주 큰 오차(오른쪽 꼬리)가 있는지 확인합니다. 큰 오차가 많으면 RMSE 가 커집니다.
display(err["abs_error"].describe())

plt.figure(figsize=(8, 4))
plt.hist(err["abs_error"], bins=50)  # bins: 값 구간을 50개로 나눠 막대로 표현
plt.title("Distribution of abs_error (v2_original)")
plt.xlabel("abs_error"); plt.ylabel("count")
plt.tight_layout(); plt.show()


## 14. 시간대별 오차 확인

버스 도착 시간은 시간대에 영향을 받을 수 있습니다. 출근 시간과 퇴근 시간에는 도로 상황이 복잡해질 수 있고, 낮 시간이나 밤 시간에는 상대적으로 흐름이 다를 수 있습니다.


In [ ]:
# 시간대(now_hour)별 평균 오차를 봅니다. 출퇴근처럼 특정 시간대에 오차가 큰지 확인하기 위해서입니다.
# count 도 함께 보는 이유: 데이터가 적은 시간대는 평균이 들쭉날쭉할 수 있으므로 표본 수를 같이 봐야 합니다.
hour_error = err.groupby("now_hour")["abs_error"].agg(["count", "mean"]).sort_index()
display(hour_error)

plt.figure(figsize=(9, 4))
plt.bar(hour_error.index.astype(int).astype(str), hour_error["mean"])
plt.title("Mean abs_error by now_hour (v2_original)")
plt.xlabel("now_hour"); plt.ylabel("mean abs_error")
plt.tight_layout(); plt.show()


## 15. 이상치 상위 1% 제외 실험

이상치란 일반적인 데이터 패턴에서 벗어난 값을 말합니다. 이번 노트북에서는 target인 `next_arrive_time`의 상위 1% 값을 매우 긴 도착 시간으로 보고 제외하는 실험을 합니다.

> ❌ 이상치를 제거하면 MAE나 RMSE가 좋아 보일 수 있습니다. 하지만 이는 어려운 데이터를 제외했기 때문일 수 있습니다. 실제 버스 도착 시간 예측 서비스에서는 큰 지연 상황도 중요합니다. 따라서 이상치 제거는 무조건 좋은 방법이 아니라 목적에 따라 판단해야 합니다.

### 이상치 제거 실험을 직접 비교하면 안 되는 이유
이상치 상위 1%를 제외한 실험은 원본 데이터 전체를 사용한 기준 모델과 평가 조건이 다릅니다. 비유하면, 어려운 문제를 시험에서 제외한 뒤 평균 점수가 올라간 것과 비슷합니다. 점수는 올라갈 수 있지만, 실제 실력이 좋아졌는지는 별도로 판단해야 합니다.

그래서 이 실험 결과에는 `directly_comparable=False`를 표시합니다. 이 표시는 "참고는 할 수 있지만, 원본 기준 모델과 단순 비교하면 안 된다"는 의미입니다.


In [ ]:
# quantile(0.99): 상위 1% 경계값을 구합니다. 이 값보다 큰 도착 시간을 "아주 긴 지연"으로 보고 제외합니다.
upper_1pct = df_model[target_col].quantile(0.99)
df_no_outlier = df_model[df_model[target_col] <= upper_1pct].copy()
removed = len(df_model) - len(df_no_outlier)
print(f"상위 1% 경계값: {upper_1pct:.2f} / 제거 {removed}건 ({removed/len(df_model)*100:.2f}%)")

# 데이터 자체가 달라졌으므로(행이 줄었으므로) 이 데이터에 맞는 별도의 split 을 새로 만듭니다.
tr_no, te_no = train_test_split(df_no_outlier.index, test_size=0.2, random_state=42)

# 같은 v2 feature 로 학습하되, directly_comparable=False 로 표시합니다.
# 평가에 쓰는 test set 에서 큰 값이 빠졌기 때문에, 기준 모델과 점수를 직접 비교하면 안 됩니다.
v2_no_outlier_result = train_xgb(
    model_name="v2_without_top_1pct_target", experiment="v2 feature (동일)",
    numeric_features=numeric_features_v2, categorical_features=categorical_features_v2,
    train_idx=tr_no, test_idx=te_no, data_df=df_no_outlier,
    directly_comparable=False,  # 평가 대상 데이터가 달라 직접 우열 비교 불가
    note="평가 대상이 달라 직접 비교 주의",
)


## 16. station_segment 추가 실험

`station_segment` 실험은 기준 모델에 정류장 구간 feature를 추가하는 실험입니다. 기준 모델에는 `now_station`과 `next_station`이 따로 들어갑니다. 하지만 이 둘을 따로 보는 것과 "현재 정류장 → 다음 정류장"을 하나의 구간으로 보는 것은 다릅니다.

### 확인할 지표와 주의점
- **MAE가 줄었는가?** 평균 절대 오차가 감소했다면 구간 feature가 도움이 되었을 가능성이 있습니다.
- **RMSE가 줄었는가?** 큰 오차도 줄었는지 확인합니다.
- **feature 수가 얼마나 늘었는가?** 구간 종류가 많으면 One-Hot Encoding 이후 feature 수가 크게 증가합니다. feature 수가 늘어난다고 항상 성능이 좋아지는 것은 아닙니다.


In [ ]:
# v2 feature 에 station_segment(정류장 구간) 하나만 추가합니다.
# 다른 조건(숫자형 feature, split, 모델 설정)은 그대로 두어야, 성능 변화가 이 feature 때문인지 알 수 있습니다.
numeric_features_station = ["distance", "day", "dayofweek", "now_hour"]
categorical_features_station = ["route_nm", "now_station", "next_station", "station_segment"]

v3_result = train_xgb(
    model_name="v3_with_station_segment", experiment="v2 + station_segment",
    numeric_features=numeric_features_station, categorical_features=categorical_features_station,
    train_idx=train_idx, test_idx=test_idx, data_df=df_model,
    note="정류장 구간 feature 추가, feature 수 증가",
)


In [ ]:
# station_segment 는 고유값이 많아 One-Hot 변환 후 컬럼 수가 크게 늘어납니다.
# 전처리 후 실제 feature 수를 비교해, 모델 입력이 얼마나 복잡해졌는지 확인합니다.
# get_feature_names_out() 이 환경에 따라 실패할 수 있으므로 try-except 로 감쌉니다. (실패해도 전체 실행은 계속)
try:
    n2 = len(v2_result["pipeline"].named_steps["preprocessor"].get_feature_names_out())
    n3 = len(v3_result["pipeline"].named_steps["preprocessor"].get_feature_names_out())
    print(f"v2_original 전처리 후 feature 수: {n2}")
    print(f"v3_with_station_segment 전처리 후 feature 수: {n3}  (증가: {n3 - n2})")
except Exception as e:
    print("feature 수 확인 중 문제가 발생했습니다:", e)


### station_segment 결과 해석 기준

`station_segment`를 추가한 뒤에는 기준 모델과 비교합니다. 결과를 해석할 때는 단정적인 표현을 피하고, 조건부로 설명하는 것이 중요합니다.

| 상황 | 적절한 해석 |
|---|---|
| MAE, RMSE 감소 | station_segment가 오차 감소에 도움이 되었을 가능성이 있습니다. |
| 성능 차이가 매우 작음 | feature 추가 효과가 크다고 단정하기 어렵습니다. |
| feature 수 크게 증가 | 모델이 더 복잡해졌다는 점도 고려해야 합니다. |


## 17. dist_segment_name 위치 권역 실험

`dist_segment_name` 실험은 위치 권역 정보를 feature로 추가하는 실험입니다. 정류장명은 매우 세부적인 정보이지만, 위치 권역은 정류장의 대략적인 지리적 위치를 나타냅니다.

- 지역 흐름 반영: 정류장 이름만으로는 파악하기 어려운 지역 흐름을 반영할 수 있습니다.
- 이동 관계 표현: 현재 권역과 다음 권역의 이동 관계를 모델에 알려 줄 수 있습니다.
- 지리 정보 단순화: 지리적 정보를 범주형 feature로 단순화할 수 있습니다.

> ⚠️ 이미 now_station, next_station, route_nm에 위치 정보가 어느 정도 포함되어 있을 수 있습니다. 이 경우 dist_segment_name은 기존 feature와 정보가 중복될 수 있습니다. 위치 권역 feature도 항상 성능을 높인다고 단정할 수는 없습니다. 따라서 결과는 반드시 비교표를 통해 확인해야 합니다.


In [ ]:
# v2 feature 에 dist_segment_name(출발권역 → 도착권역) 추가합니다.
# now_dist_to_*, next_dist_to_* 같은 거리 숫자 컬럼도 feature 에 추가 합니다
numeric_features_dist = [
    "distance", "day", "dayofweek", "now_hour",
    "now_dist_to_up", "now_dist_to_down", "now_dist_to_right", "now_dist_to_center",
    "next_dist_to_up", "next_dist_to_down", "next_dist_to_right", "next_dist_to_center"
]
categorical_features_dist = ["route_nm", "now_station", "next_station", "dist_segment_name"]

v2_dist_result = train_xgb(
    model_name="v2_with_dist_segment_name", experiment="v2 + dist_segment_name",
    numeric_features=numeric_features_dist, categorical_features=categorical_features_dist,
    train_idx=train_idx, test_idx=test_idx, data_df=df_model,
    note="위치 권역 조합 feature 추가",
)


## 18. 전체 실험 결과 비교

노트북 마지막에서는 여러 실험 결과를 하나의 표로 모읍니다. 이 비교표가 이번 노트북의 핵심 결과물입니다.

> 실제 수치는 노트북을 실행한 결과를 기준으로 비교하세요. MAE와 RMSE는 작을수록 좋습니다.

> ❌ 모든 실험을 같은 방식으로 비교하면 안 됩니다. 특히 이상치 제거 실험은 평가 데이터 조건이 달라졌기 때문에 기준 모델과 직접 비교하면 안 됩니다.

### directly_comparable의 의미
노트북의 평가 결과에는 `directly_comparable`이라는 값이 있습니다. 이 값은 해당 실험 결과를 기준 모델과 직접 비교해도 되는지 표시합니다.

- **True**: feature만 추가하고 train/test split과 데이터 조건이 같다면 기준 모델과 비교하기 쉽습니다. 이 경우 직접 비교가 가능합니다. (예: `v3_with_station_segment`, `v2_with_dist_segment_name`)
- **False**: 이상치 상위 1%를 제외한 실험은 평가 대상 데이터 자체가 달라집니다. 원본 데이터 전체를 대상으로 한 기준 모델과 단순 비교하면 공정하지 않습니다. (예: `v2_without_top_1pct_target`)


In [ ]:
# 네 실험의 평가 결과(딕셔너리)를 하나의 표로 모읍니다.
# directly_comparable 이 False 인 모델(상위 1% 제외)은 평가 대상 데이터가 달라
# 기준 모델과 단순 점수 비교를 하면 안 됩니다.
comparison_df = pd.DataFrame([
    v2_result["evaluation"],
    v2_no_outlier_result["evaluation"],
    v3_result["evaluation"],
    v2_dist_result["evaluation"],
])
comparison_df[["model_name", "experiment", "directly_comparable", "MAE", "RMSE", "note"]]


In [ ]:
# directly_comparable=True 인 실험만 모읍니다.
# 평가 조건(test set)이 같은 실험끼리만 MAE 기준으로 정렬해 상대적으로 좋은 모델을 봅니다.
# (상위 1% 제외 실험은 평가 대상이 달라 여기서 빠집니다.)
comparable_df = comparison_df[comparison_df["directly_comparable"] == True].sort_values("MAE")
comparable_df[["model_name", "experiment", "MAE", "RMSE", "note"]]


## 19. 그래프로 결과 비교하기

표는 정확한 숫자를 확인하는 데 좋고, 그래프는 전체 차이를 빠르게 파악하는 데 좋습니다. 모델별 MAE와 RMSE를 막대그래프로 비교합니다.

### 그래프를 볼 때 확인할 것
- 어떤 모델의 MAE가 가장 낮은가? (낮을수록 평균 오차가 작습니다.)
- 직접 비교 가능한 실험인가? (데이터 조건이 다른 실험은 단순 순위 비교를 하면 안 됩니다.)
- 차이가 큰가, 작은가? (차이가 매우 작다면 실제 의미가 크지 않을 수 있습니다.)
- feature 수가 지나치게 늘지는 않았는가? (성능 향상이 복잡도 증가를 감수할 만큼 의미 있는지 확인합니다.)


In [ ]:
# 전체 모델 MAE / RMSE 비교 그래프 (작을수록 좋음)
names = comparison_df["model_name"].tolist()
x = np.arange(len(names)); width = 0.35

plt.figure(figsize=(11, 4))
plt.bar(x - width/2, comparison_df["MAE"], width, label="MAE")
plt.bar(x + width/2, comparison_df["RMSE"], width, label="RMSE")
plt.title("All experiments: MAE / RMSE")
plt.xlabel("model")
plt.ylabel("error (Lower is better)")
plt.xticks(x, names, rotation=20, ha="right")
plt.legend()
plt.tight_layout(); plt.show()


## 20. 기준 모델 대비 변화량 해석

기준 모델 대비 변화량을 계산하면 feature 추가가 어떤 영향을 주었는지 더 쉽게 볼 수 있습니다.

### 변화량 해석 시 함께 고려할 것
1. 지표가 얼마나 변했는가 (작은 변화는 우연일 수 있습니다.)
2. 같은 데이터 조건에서 비교했는가 (조건이 다르면 직접 비교 불가.)
3. feature 수가 얼마나 늘었는가 (복잡도 증가 여부 확인.)
4. EDA 결과와 연결해서 설명할 수 있는가 (근거 있는 해석이 중요합니다.)
5. 실제 서비스 관점에서도 의미가 있는가 (숫자가 좋아졌다고 항상 실용적인 것은 아닙니다.)


In [ ]:
# 기준 모델(v2_original) 대비 성능 차이를 계산합니다.
# directly_comparable=True 인 모델만 대상으로 합니다. (이상치 제외 모델은 평가 조건이 달라 자동 제외됩니다.)
ci = comparison_df.set_index("model_name")
base = ci.loc["v2_original"]
targets = comparison_df[(comparison_df["directly_comparable"]) &
                         (comparison_df["model_name"] != "v2_original")]["model_name"].tolist()

# 다만 차이가 아주 작으면 의미를 크게 두지 말고, EDA 결과와 연결해 신중하게 해석합니다.
rows = [{
    "model_name": name,
    "MAE_diff_vs_v2": ci.loc[name, "MAE"] - base["MAE"],
    "RMSE_diff_vs_v2": ci.loc[name, "RMSE"] - base["RMSE"]
} for name in targets]

print("차이 계산 대상:", targets)
print("차이 계산 제외(평가 조건 다름):", ["v2_without_top_1pct_target"])
pd.DataFrame(rows)


## 21. 이번 노트북의 핵심 정리

- **기준 모델은 반드시 필요합니다.** 기준 모델이 있어야 feature를 추가했을 때 좋아졌는지 나빠졌는지 비교할 수 있습니다.
- **평균 성능 지표만으로는 충분하지 않습니다.** 모델이 어떤 상황에서 많이 틀리는지 확인하려면 오차 분석이 필요합니다.
- **이상치 제거는 조심해야 합니다.** 지표가 좋아질 수 있지만, 실제 서비스에서 중요한 어려운 데이터를 제외한 결과일 수 있습니다.
- **feature 추가가 항상 성능을 높이지는 않습니다.** feature 수 증가, 정보 중복, 모델 복잡도 증가를 함께 고려해야 합니다.
- **실험 결과는 같은 조건에서 비교해야 합니다.** 데이터 조건이 달라진 실험은 직접 비교하면 안 됩니다.

### Feature Engineering을 해석하는 방법
좋은 feature engineering은 EDA에서 발견한 가설을 검증하는 과정입니다.

`EDA 관찰 → 오류 가정 → 설명 가능한 특성 → 동일조건 재학습 → 지표·오차 비교`

이번 노트북의 `station_segment`와 `dist_segment_name` 실험도 이 흐름을 보여줍니다. 중요한 점은 feature 추가가 항상 성능 향상으로 이어지지는 않는다는 것입니다. 때로는 정보가 중복되거나, feature 수만 늘어나거나, 모델이 더 복잡해질 수 있습니다.

### 다음 단계: 하이퍼파라미터 튜닝
이번 노트북에서는 feature 실험에 집중했습니다. 모델의 하이퍼파라미터는 크게 바꾸지 않고 고정했습니다. 다음 노트북(`jeju_bus_xgboost_hyperparameter_tuning.ipynb`)에서는 feature를 어느 정도 정리한 뒤, 모델이 그 정보를 어떻게 학습하게 할 것인지(`n_estimators`, `max_depth`, `learning_rate` 등) 탐구합니다.
